In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import gradio as gr

# ===== 1. 讀檔 =====
winner = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')

# ===== 2. 年份補充與欄位準備 =====
winner['year'] = pd.to_datetime(winner['Date']).dt.year
winner['year_raw'] = winner['year']
winner['Grand Prix raw'] = winner['Grand Prix']

# ===== 3. 合併資料 =====
df = winner.merge(
    drivers[['Driver', 'Car', 'year', 'Nationality', 'PTS']],
    left_on=['Winner', 'Car', 'year'],
    right_on=['Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_driver')
)
df = df.merge(
    teams[['Team', 'PTS', 'year']],
    left_on=['Car', 'year'],
    right_on=['Team', 'year'],
    how='left',
    suffixes=('', '_team')
)

df['PTS'] = df['PTS'].fillna(0)
df['PTS_team'] = df['PTS_team'].fillna(0)

train_cols = ['year', 'Grand Prix', 'Car', 'Nationality', 'PTS', 'PTS_team']
X = df[train_cols].copy()
y = df['Winner'].copy()

# ===== 4. 只保留多於一次冠軍車手（避免 label error，建議依你需求調整）=====
value_counts = y.value_counts()
valid_drivers = value_counts[value_counts >= 2].index
mask = y.isin(valid_drivers)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)
df = df[mask].reset_index(drop=True)

# ===== 5. 全量 LabelEncoder（fit 全部出現過的類別）=====
cat_cols = ['Grand Prix', 'Car', 'Nationality']
encoders = {}
for col in cat_cols:
    all_cats = pd.concat([
        df[col], 
        drivers[col] if col in drivers.columns else pd.Series(), 
        teams['Team'] if col == 'Car' and 'Team' in teams.columns else pd.Series()
    ]).drop_duplicates().astype(str)
    le = LabelEncoder()
    le.fit(all_cats)
    X[col] = le.transform(X[col].astype(str))
    encoders[col] = le

# y (Winner) 也 fit 全部出現過的車手
all_drivers = pd.concat([
    df['Winner'],
    drivers['Driver']
]).drop_duplicates().astype(str)
le_winner = LabelEncoder()
le_winner.fit(all_drivers)
y_enc = le_winner.transform(y.astype(str))
encoders['Winner'] = le_winner

# ===== 6. 數值標準化 =====
num_cols = ['year', 'PTS', 'PTS_team']
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# ===== 7. PyTorch Dataset =====
class F1RaceSet(Dataset):
    def __init__(self, X, y):
        self.X_cat = torch.tensor(X[cat_cols].values, dtype=torch.long)
        self.X_num = torch.tensor(X[num_cols].values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

batch_size = 128
trainset = F1RaceSet(X_train, y_train)
testset = F1RaceSet(X_test, y_test)
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testset, batch_size=batch_size)

# ===== 8. Model =====
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_features, embedding_dim=8, hidden_dim=128, num_classes=None):
        super().__init__()
        self.emb_layers = nn.ModuleList([
            nn.Embedding(cat_dim, embedding_dim) for cat_dim in cat_dims
        ])
        input_dim = embedding_dim * len(cat_dims) + num_num_features
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cat_dims = [len(encoders[col].classes_) for col in cat_cols]
num_classes = len(encoders['Winner'].classes_)
model = F1DNN(cat_dims, len(num_cols), embedding_dim=8, hidden_dim=128, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ===== 9. 訓練 =====
epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_cat_batch, X_num_batch, y_batch in train_loader:
        X_cat_batch, X_num_batch, y_batch = X_cat_batch.to(device), X_num_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_cat_batch, X_num_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    avg_loss = total_loss / len(trainset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# ===== 10. 測試集驗證與預測展示 + 準確率 =====

test_idx = X_test.index.values
test_df = df.iloc[test_idx].reset_index(drop=True)
winner_decoder = encoders['Winner']

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_cat_batch, X_num_batch, y_batch in test_loader:
        X_cat_batch, X_num_batch = X_cat_batch.to(device), X_num_batch.to(device)
        logits = model(X_cat_batch, X_num_batch)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y_batch.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

acc = accuracy_score(all_labels, all_preds)
print(f"\nTest Accuracy: {acc:.4f}\n")

np.random.seed(42)
rand_idx = np.random.choice(len(test_df), size=10, replace=False)
for i in rand_idx:
    row = test_df.iloc[i]
    year = int(row['year_raw'])
    grand_prix = row['Grand Prix raw']
    true_winner = winner_decoder.inverse_transform([all_labels[i]])[0]
    pred_winner = winner_decoder.inverse_transform([all_preds[i]])[0]
    print(f"{year} {grand_prix}\nPredicted Winner: {pred_winner}  , True Winner: {true_winner} \n")

# ===== 11. Gradio 介面預測未來賽事 =====

all_years = sorted(df['year_raw'].unique())
future_years = [y for y in range(all_years[-1] + 1, all_years[-1] + 3)]
all_years_extended = all_years + future_years
all_gps = sorted(df['Grand Prix raw'].unique())

def get_prev_season_driverlist(year):
    last_year = year - 1
    driver_groups = drivers[drivers['year'] == last_year].copy()
    driver_groups = driver_groups[['Driver', 'Car', 'Nationality', 'PTS']].drop_duplicates()
    team_pts = teams[teams['year'] == last_year][['Team', 'PTS']].rename(columns={'Team': 'Car', 'PTS': 'PTS_team'})
    driver_groups = driver_groups.merge(team_pts, on='Car', how='left')
    driver_groups['PTS_team'] = driver_groups['PTS_team'].fillna(0)
    driver_groups = driver_groups.rename(columns={'Driver': 'Winner'})
    return driver_groups

def predict_future_all(year, grand_prix):
    driver_list = get_prev_season_driverlist(year)
    if driver_list is None or driver_list.empty:
        return f"查無 {year-1} 年所有參賽車手資料，無法預測。"

    model.eval()
    all_results = []
    for _, row in driver_list.iterrows():
        try:
            feat = {}
            feat['year'] = year
            feat['Grand Prix'] = encoders['Grand Prix'].transform([grand_prix])[0]
            feat['Car'] = encoders['Car'].transform([row['Car']])[0]
            feat['Nationality'] = encoders['Nationality'].transform([row['Nationality']])[0]
            feat['PTS'] = row['PTS']
            feat['PTS_team'] = row['PTS_team']

            num_arr = np.array([[feat['year'], feat['PTS'], feat['PTS_team']]])
            num_arr = scaler.transform(num_arr)
            cat_arr = np.array([[feat['Grand Prix'], feat['Car'], feat['Nationality']]])

            cat_tensor = torch.tensor(cat_arr, dtype=torch.long).to(device)
            num_tensor = torch.tensor(num_arr, dtype=torch.float32).to(device)

            with torch.no_grad():
                logits = model(cat_tensor, num_tensor)
                prob = torch.softmax(logits, dim=1).cpu().numpy()[0]

            # 若該車手沒在模型類別也能編碼，但預測分數僅供參考
            driver_idx = encoders['Winner'].transform([row['Winner']])[0]
            driver_win_prob = prob[driver_idx]
            all_results.append((driver_win_prob, row['Winner'], row['Car']))
        except Exception as e:
            print(f"特徵編碼錯誤：{row['Winner']}（車隊：{row['Car']}）原因：{e}")
            continue

    if not all_results:
        return "無法對任何參賽者進行預測（可能有新車隊/國籍等特徵編碼問題）"

    all_results.sort(reverse=True)
    out_txt = f"【{year} {grand_prix} 奪冠預測（用 {year-1} 年全年度所有參賽車手）】\n"
    for i, (prob, driver, car) in enumerate(all_results[:10]):
        out_txt += f"{i+1}. {driver}（車隊：{car}）→ 冠軍機率：{prob:.3f}\n"
    return out_txt

iface = gr.Interface(
    fn=predict_future_all,
    inputs=[
        gr.Dropdown(choices=all_years_extended, label="年份（支援預測未來）"),
        gr.Dropdown(choices=all_gps, label="分站名（Grand Prix）")
    ],
    outputs=gr.Textbox(label="預測結果"),
    title="F1 賽事冠軍預測（完整參賽名單，支援未來）",
    description="選擇年份與分站，系統自動用前一年所有參賽車手進行冠軍預測。"
)
iface.launch()


Epoch 1/30, Loss: 5.9959
Epoch 2/30, Loss: 5.3591
Epoch 3/30, Loss: 4.8630
Epoch 4/30, Loss: 4.4538
Epoch 5/30, Loss: 4.0952
Epoch 6/30, Loss: 3.7786
Epoch 7/30, Loss: 3.5059
Epoch 8/30, Loss: 3.2331
Epoch 9/30, Loss: 2.9794
Epoch 10/30, Loss: 2.7216
Epoch 11/30, Loss: 2.4504
Epoch 12/30, Loss: 2.2543
Epoch 13/30, Loss: 2.0685
Epoch 14/30, Loss: 1.8981
Epoch 15/30, Loss: 1.7291
Epoch 16/30, Loss: 1.5990
Epoch 17/30, Loss: 1.4756
Epoch 18/30, Loss: 1.3789
Epoch 19/30, Loss: 1.2589
Epoch 20/30, Loss: 1.1664
Epoch 21/30, Loss: 1.1089
Epoch 22/30, Loss: 1.0427
Epoch 23/30, Loss: 1.0030
Epoch 24/30, Loss: 0.9156
Epoch 25/30, Loss: 0.8852
Epoch 26/30, Loss: 0.8204
Epoch 27/30, Loss: 0.7844
Epoch 28/30, Loss: 0.7511
Epoch 29/30, Loss: 0.7148
Epoch 30/30, Loss: 0.6933

Test Accuracy: 0.8140

1987 Austria
Predicted Winner: Nigel  Mansell   , True Winner: Nigel  Mansell  

1984 Great Britain
Predicted Winner: Niki  Lauda   , True Winner: Niki  Lauda  

1954 Italy
Predicted Winner: Juan Manuel  F

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with f

特徵編碼錯誤：Louis Rosier （車隊：nan）原因：y contains previously unseen labels: nan
